<a href="https://colab.research.google.com/github/Zarrialvi/Airport-Recognition-system/blob/main/Airport.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install face_recognition opencv-python cryptography



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=2e96e7cf17e7d0e260c1528d85470d5f1882314ffe676075e53d6226b81d6ab0
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models


In [ ]:
import face_recognition
import cv2
import numpy as np
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os, base64, datetime, json
from google.colab import files

# In‑memory "database" for demo (passenger_id -> encrypted_template)
TEMPLATES_DB = {}
AUDIT_LOGS = []

def log_event(event_type, passenger_id=None, details=""):
    AUDIT_LOGS.append({
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "event_type": event_type,
        "passenger_id": passenger_id,
        "details": details,
    })


In [ ]:
def derive_key_from_pin(pin: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        iterations=200_000,
    )
    return kdf.derive(pin.encode())

def encrypt_embedding(embedding: np.ndarray, pin: str) -> str:
    salt = os.urandom(16)
    key = derive_key_from_pin(pin, salt)
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)
    data = embedding.astype(np.float64).tobytes()
    ct = aesgcm.encrypt(nonce, data, None)
    blob = base64.b64encode(salt + nonce + ct).decode()
    return blob

def decrypt_embedding(blob: str, pin: str) -> np.ndarray:
    raw = base64.b64decode(blob.encode())
    salt, nonce, ct = raw[:16], raw[16:28], raw[28:]
    key = derive_key_from_pin(pin, salt)
    aesgcm = AESGCM(key)
    data = aesgcm.decrypt(nonce, ct, None)
    emb = np.frombuffer(data, dtype=np.float64)
    return emb

def get_face_embedding(image_path: str) -> np.ndarray:
    img = face_recognition.load_image_file(image_path)
    locations = face_recognition.face_locations(img)
    if not locations:
        raise ValueError("No face found in image")
    encodings = face_recognition.face_encodings(img, known_face_locations=locations)
    if not encodings:
        raise ValueError("Could not compute face encoding")
    return encodings[0]


In [ ]:
# ==== ENROLMENT FLOW ====
print("=== Enrolment ===")
passenger_id = input("Enter passenger ID (e.g., PAX001): ").strip()
pin = input("Set PIN for this passenger (numbers or string): ").strip()

if not passenger_id or not pin:
    raise ValueError("Passenger ID and PIN are required")

print("\nUpload enrolment image (clear, front face)...")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No image uploaded")
enrol_image_path = list(uploaded.keys())[0]

try:
    enrol_embedding = get_face_embedding(enrol_image_path)
except Exception as e:
    raise RuntimeError(f"Error processing enrolment image: {e}")

encrypted_template = encrypt_embedding(enrol_embedding, pin)

# Store in "DB"
TEMPLATES_DB[passenger_id] = encrypted_template
log_event("enrolment", passenger_id, f"Template enrolled from image {enrol_image_path}")

print("\nEnrolment successful.")
print("Stored passengers in DB:", list(TEMPLATES_DB.keys()))


=== Enrolment ===
Enter passenger ID (e.g., PAX001): PAX0001
Set PIN for this passenger (numbers or string): 1234

Upload enrolment image (clear, front face)...


Saving P9.png to P9.png


RuntimeError: Error processing enrolment image: No face found in image

In [ ]:
# ==== AUTHENTICATION FLOW ====
print("=== Authentication ===")
if not TEMPLATES_DB:
    raise RuntimeError("No passengers enrolled yet. Run enrolment cell first.")

auth_passenger_id = input("Enter passenger ID for authentication: ").strip()
pin_auth = input("Enter PIN: ").strip()

if auth_passenger_id not in TEMPLATES_DB:
    log_event("auth_failed", auth_passenger_id, "Passenger ID not found")
    raise RuntimeError("Passenger ID not found. Enrol first.")

encrypted_template = TEMPLATES_DB[auth_passenger_id]

print("\nUpload authentication image (checkpoint camera capture)...")
uploaded_auth = files.upload()
if not uploaded_auth:
    raise ValueError("No authentication image uploaded")
auth_image_path = list(uploaded_auth.keys())[0]

try:
    stored_template = decrypt_embedding(encrypted_template, pin_auth)
except Exception as e:
    log_event("auth_failed", auth_passenger_id, f"Decryption error: {e}")
    raise RuntimeError("PIN incorrect or decryption failed")

try:
    auth_embedding = get_face_embedding(auth_image_path)
except Exception as e:
    raise RuntimeError(f"Error processing authentication image: {e}")

dist = np.linalg.norm(stored_template - auth_embedding)
threshold = 0.6  # tune depending on testing

print("Distance:", dist)
if dist < threshold:
    print("MATCH: Passenger verified, allow boarding/checkpoint pass.")
    log_event("auth_success", auth_passenger_id, f"Distance={dist:.4f}")
else:
    print("NO MATCH: Route to manual ID verification.")
    log_event("auth_no_match", auth_passenger_id, f"Distance={dist:.4f}")


=== Authentication ===


RuntimeError: No passengers enrolled yet. Run enrolment cell first.

In [ ]:
print("=== Stored templates (keys only) ===")
print(list(TEMPLATES_DB.keys()))

print("\n=== Audit logs ===")
print(json.dumps(AUDIT_LOGS, indent=2))


=== Stored templates (keys only) ===
[]

=== Audit logs ===
[]
